# watsonx.governance Setup: Prompt Templates & Observability

This notebook demonstrates how to:
1. Create and manage detached prompt template assets in watsonx.governance
2. Configure OpenScale monitoring for AI quality evaluation
3. Set up LangFuse tracing for agent observability (optional)
4. Evaluate prompt performance with governance metrics

**Runtime Requirements:** Python 3.10 or greater

**Note:** Search for `# EDIT THIS` to find required user inputs

## Prerequisites

- IBM watsonx.ai project and deployment space
- IBM Watson OpenScale service credentials
- IBM Cloud API key
- (Optional) LangFuse account for tracing
- (Optional) Test data CSV for evaluation

## Table of Contents

1. [Setup & Installation](#setup)
2. [Create Prompt Template Asset](#prompt-template)
3. [Configure OpenScale Monitoring](#openscale)
4. [Run Risk Evaluations](#evaluate)
5. [View Governance Metrics](#metrics)
6. [Optional: LangFuse Tracing Setup](#langfuse)
7. [View Factsheets](#factsheets)

## 1. Setup & Installation <a name="setup"></a>

### Install Required Packages

In [ ]:
!pip install ibm-watson-openscale>=4.0.0
!pip install ibm-watsonx-ai>=1.1.0
!pip install datasets>=2.10.0
!pip install evaluate>=0.4.0

### Import Libraries

In [ ]:
import os
import json
import time
import pandas as pd
from ibm_watson_openscale import APIClient as OpenScaleClient
from ibm_watson_openscale.supporting_classes.enums import *
from ibm_watsonx_ai import APIClient as WMLClient
from ibm_watsonx_ai import Credentials

### Configure Credentials

In [ ]:
# EDIT THIS - Add your credentials
IAM_URL = "https://iam.cloud.ibm.com"
DATAPLATFORM_URL = "https://api.dataplatform.cloud.ibm.com"
OPENSCALE_URL = "https://api.aiopenscale.cloud.ibm.com"

CLOUD_API_KEY = "your-cloud-api-key-here"  # EDIT THIS

WML_CREDENTIALS = {
    "url": "https://us-south.ml.cloud.ibm.com",  # Update region if needed
    "apikey": CLOUD_API_KEY
}

### Set Project and Space IDs

In [ ]:
# EDIT THIS - Add your project and space IDs
PROJECT_ID = "your-project-id-here"  # Project where prompt template will be created
SPACE_ID = "your-space-id-here"      # Deployment space for monitoring

### Initialize Clients

In [ ]:
# Initialize watsonx.ai client
wml_client = WMLClient(
    credentials=Credentials(
        url=WML_CREDENTIALS["url"],
        api_key=WML_CREDENTIALS["apikey"]
    ),
    project_id=PROJECT_ID
)

print("watsonx.ai client initialized")

In [ ]:
# Initialize OpenScale client
wos_client = OpenScaleClient(
    service_url=OPENSCALE_URL,
    authenticator={
        "url": IAM_URL,
        "apikey": CLOUD_API_KEY
    }
)

print("Watson OpenScale client initialized")

## 2. Create Prompt Template Asset <a name="prompt-template"></a>

### Define Your Prompt Template

Create a detached prompt template that will be monitored for quality and risk.

In [ ]:
# EDIT THIS - Customize your prompt template
prompt_template_name = "example_prompt_template"

prompt_text = """
You are a helpful AI assistant.

Context: {context}

Question: {question}

Please provide a clear and accurate answer based on the context provided.

Answer:
"""

# Prompt template metadata
prompt_metadata = {
    wml_client.repository.PromptTemplateMetaNames.NAME: prompt_template_name,
    wml_client.repository.PromptTemplateMetaNames.DESCRIPTION: "Generic prompt template for Q&A tasks",
    wml_client.repository.PromptTemplateMetaNames.TASK_IDS: ["generation"],  # Can be: generation, summarization, classification, etc.
    wml_client.repository.PromptTemplateMetaNames.INPUT_MODE: "structured",
    wml_client.repository.PromptTemplateMetaNames.MODEL_ID: "meta-llama/llama-3-3-70b-instruct",
    wml_client.repository.PromptTemplateMetaNames.MODEL_PARAMETERS: {
        "decoding_method": "greedy",
        "max_new_tokens": 500,
        "temperature": 0.7,
        "repetition_penalty": 1.0
    }
}

print("Prompt template configured")

### Store Prompt Template in Project

In [ ]:
# Store the prompt template
stored_prompt_template = wml_client.repository.store_prompt_template(
    prompt_template=prompt_text,
    meta_props=prompt_metadata
)

prompt_template_id = wml_client.repository.get_prompt_template_id(stored_prompt_template)

print(f"Prompt template stored with ID: {prompt_template_id}")

## 3. Configure OpenScale Monitoring <a name="openscale"></a>

### Set Up Data Mart

In [ ]:
# Get or create data mart
data_marts = wos_client.data_marts.list().result.data_marts

if len(data_marts) == 0:
    # Create new data mart
    data_mart_details = wos_client.data_marts.add(
        background_mode=False,
        name="WatsonOpenScale Data Mart",
        description="Data mart for governance monitoring",
        internal_database=True
    ).result
    data_mart_id = data_mart_details.metadata.id
else:
    data_mart_id = data_marts[0].metadata.id

print(f"Using data mart ID: {data_mart_id}")

### Subscribe Prompt Template for Monitoring

In [ ]:
# Create subscription for the prompt template
subscription_details = wos_client.subscriptions.add(
    data_mart_id=data_mart_id,
    service_provider_id=wos_client.service_providers.get_wml_provider_id(),
    asset={
        "asset_id": prompt_template_id,
        "asset_type": "prompt_template",
        "name": prompt_template_name,
        "input_data_type": "unstructured_text",
        "problem_type": "generation"  # Can be: generation, summarization, question_answering, etc.
    },
    deployment={
        "deployment_type": "online",
        "name": f"{prompt_template_name}_deployment"
    },
    asset_properties={
        "label_column": "expected_output",  # EDIT THIS - column name for ground truth
        "prediction_column": "generated_output"  # EDIT THIS - column name for model output
    }
).result

subscription_id = subscription_details.metadata.id
print(f"Subscription created with ID: {subscription_id}")

### Configure Quality Monitors

In [ ]:
# Configure generative AI quality monitoring
quality_monitor = wos_client.monitor_instances.create(
    data_mart_id=data_mart_id,
    background_mode=False,
    monitor_definition_id=wos_client.monitor_definitions.MONITORS.GENERATIVE_AI_QUALITY.ID,
    target={
        "target_type": "subscription",
        "target_id": subscription_id
    },
    parameters={
        "min_feedback_data_size": 10,  # Minimum records needed for evaluation
        "metrics": [
            {"name": "rouge_score", "enabled": True, "threshold": 0.7},
            {"name": "bleu", "enabled": True, "threshold": 0.5},
            {"name": "meteor", "enabled": True, "threshold": 0.6},
            {"name": "flesch_reading_ease", "enabled": True},
            {"name": "sentence_coherence", "enabled": True}
        ]
    }
).result

print("Quality monitor configured")

## 4. Run Risk Evaluations <a name="evaluate"></a>

### Prepare Test Data (Optional)

If you have test data, load it here for evaluation.

In [ ]:
# EDIT THIS - Load your test data
# Option 1: Load from CSV
# test_data = pd.read_csv('your_test_data.csv')

# Option 2: Create sample data
test_data = pd.DataFrame({
    'input': ['Sample question 1', 'Sample question 2', 'Sample question 3'],
    'expected_output': ['Expected answer 1', 'Expected answer 2', 'Expected answer 3'],
    'generated_output': ['Generated answer 1', 'Generated answer 2', 'Generated answer 3']
})

print(f"Test data loaded: {len(test_data)} records")
test_data.head()

### Send Data to OpenScale for Evaluation

In [ ]:
# Store evaluation data
payload_data_set_id = wos_client.data_sets.store_records(
    data_mart_id=data_mart_id,
    data_set_id=subscription_id,
    request_body=test_data.to_dict(orient='records'),
    background_mode=False
)

print(f"Test data stored for evaluation")

### Run Quality Evaluation

In [ ]:
# Trigger evaluation run
run_details = wos_client.monitor_instances.run(
    monitor_instance_id=quality_monitor.metadata.id,
    background_mode=False
).result

print(f"Evaluation completed")
print(f"Run ID: {run_details.metadata.id}")

## 5. View Governance Metrics <a name="metrics"></a>

### Display Quality Metrics

In [ ]:
# Retrieve and display metrics
time.sleep(5)  # Wait for metrics to be calculated

metrics = wos_client.monitor_instances.show_metrics(
    monitor_instance_id=quality_monitor.metadata.id
)

print("Quality Metrics:")
print(metrics)

### Get Detailed Metric Values

In [ ]:
# Get specific metric values
measurements = wos_client.monitor_instances.measurements.query(
    monitor_instance_id=quality_monitor.metadata.id,
    recent_count=10
).result

for measurement in measurements.to_dict()['measurements']:
    print(f"Metric: {measurement.get('metric_name')}")
    print(f"Value: {measurement.get('value')}")
    print(f"Timestamp: {measurement.get('timestamp')}")
    print("-" * 50)

## 6. Optional: LangFuse Tracing Setup <a name="langfuse"></a>

Set up LangFuse for detailed agent tracing and observability.

### Install watsonx Orchestrate SDK

In [ ]:
# For local development with virtual environment
# !python -m venv .venv
# !source .venv/bin/activate
!pip install --upgrade ibm-watsonx-orchestrate

### Configure LangFuse (Command Line)

Run these commands in your terminal:

```bash
# 1. Set up Orchestrate environment
orchestrate env add --name wxo-gov \
  --url https://api.us-south.watson-orchestrate.cloud.ibm.com/instances/YOUR_INSTANCE_ID

# 2. Activate environment
orchestrate env activate wxo-gov
# (You will be prompted for your IBM Cloud API key)

# 3. Configure LangFuse
orchestrate settings observability langfuse configure \
  --url "https://cloud.langfuse.com/api/public/otel" \
  --api-key "sk-lf-YOUR_SECRET_KEY" \
  --health-uri "https://cloud.langfuse.com" \
  --project-id "YOUR_LANGFUSE_PROJECT_ID" \
  --config-json '{"public_key": "pk-lf-YOUR_PUBLIC_KEY"}'
```

### LangFuse Setup Steps

1. Go to [https://cloud.langfuse.com](https://cloud.langfuse.com)
2. Create an organization and project
3. Click "Create an API key"
4. Save the following:
   - Secret Key (API key): `sk-lf-...`
   - Public Key: `pk-lf-...`
   - Project ID: From project settings
5. Run the configuration command above with your keys
6. Interact with your agent in Orchestrate
7. View traces in LangFuse under "Tracing"

## 7. View Factsheets <a name="factsheets"></a>

In [ ]:
# Generate factsheet URL
factsheets_url = f"https://dataplatform.cloud.ibm.com/ml-runtime/prompt-templates/{prompt_template_id}/details?project_id={PROJECT_ID}&context=wx"

print(f"View your prompt template and governance metrics here:")
print(factsheets_url)

## Additional Resources

- [Watson OpenScale Documentation](https://cloud.ibm.com/docs/watson-openscale)
- [watsonx.governance Best Practices](https://www.ibm.com/products/watsonx-governance)
- [LangFuse Documentation](https://langfuse.com/docs)
- [Prompt Template Asset Guide](https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-prompt-lab.html)